In [2]:
import os

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"
os.environ["JAX_DEFAULT_MATMUL_PRECISION"] = "highest"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1"

import argparse
import random
import shutil
import sys
from datetime import datetime
from typing import Optional

import hydra
import numpy as np
import torch
import tqdm
from omegaconf import OmegaConf

from flash_rl.agents import create_agent
from flash_rl.common import create_logger
from flash_rl.envs import create_envs
from flash_rl.evaluation import evaluate, record_video
from flash_rl.types import Tensor


### Configs

In [3]:
config_path = "./configs"
config_name = "metra_base"

overrides = [
    "num_env_steps=1000",
    "num_train_envs=1",
    "agent=metra-test"
]

from omegaconf import OmegaConf
OmegaConf.register_new_resolver("eval", lambda s: eval(s))

 # initialize config
hydra.initialize(version_base=None, config_path=config_path)
cfg = hydra.compose(config_name=config_name, overrides=overrides)
OmegaConf.resolve(cfg)


# Set random seed
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

/home/cv/zjx/FlashSAC/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


make envs

In [4]:
train_env, eval_env, record_env = create_envs(**cfg.env)

observation_space = train_env.observation_space
action_space = train_env.action_space


Using provided `max_episode_steps` (1000) instead of the environment's default (1000).
Using provided `max_episode_steps` (1000) instead of the environment's default (1000).
Using provided `max_episode_steps` (1000) instead of the environment's default (1000).


/home/cv/zjx/FlashSAC/.venv/lib/python3.11/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Ant-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


make agent

In [5]:
init_obs, env_info = train_env.reset()
agent = create_agent(
    observation_space=observation_space,
    action_space=action_space,
    env_info=env_info,
    cfg=cfg.agent,
)

/home/cv/zjx/FlashSAC/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
W0524 13:07:52.051000 446247 torch/utils/cpp_extension.py:117] [0/0] No CUDA runtime is found, using CUDA_HOME='/usr'


In [6]:
prev_transition = {
    "next_observation": init_obs
}
action = agent.sample_actions(0, prev_transition, False)

In [7]:
step_info = train_env.step(actions=action)

In [8]:

agent._skill_encoder.network(torch.from_numpy(init_obs).float(), False)

tensor([[ 0.8807, -0.4000]], grad_fn=<MmBackward0>)

In [9]:
loaded_agent = create_agent(
    observation_space=observation_space,
    action_space=action_space,
    env_info=env_info,
    cfg=cfg.agent,
)

In [10]:
ckpt_path = '/home/cv/zjx/FlashSAC/models/test/metra-normal/Ant-v4/seed0-0518-160602/step250000/skill_encoder.pt'

loaded_agent._skill_encoder.load(path=ckpt_path)

In [11]:
loaded_agent._skill_encoder.network(torch.from_numpy(init_obs).float(), False)

tensor([[-14.7577,  -2.3906]], grad_fn=<MmBackward0>)

In [ ]:
episode_length = 100
buffer = []

prev_obs, env_info = train_env.reset()
prev_transition = {
    "next_observation": init_obs
}
for _ in range(episode_length):   
    action = agent.sample_actions(0, prev_transition, False)
    obs, rew, term, trun, info = train_env.step(action)
    buffer.append((prev_obs, action))
    prev_obs = obs
    prev_transition["next_observation"] = obs


In [20]:
encoded = []
for obs, _ in buffer:
    encoded_state = loaded_agent._skill_encoder.network(torch.from_numpy(obs).float(), False)
    encoded.append(encoded_state)

[tensor([[-16.0500,  -3.4889]], grad_fn=<MmBackward0>),
 tensor([[-15.1414,  -2.8149]], grad_fn=<MmBackward0>),
 tensor([[-14.4399,  -1.8834]], grad_fn=<MmBackward0>),
 tensor([[-13.6860,  -0.9237]], grad_fn=<MmBackward0>),
 tensor([[-13.4422,  -0.2156]], grad_fn=<MmBackward0>),
 tensor([[-13.5612,  -1.3484]], grad_fn=<MmBackward0>),
 tensor([[-14.7039,  -4.5057]], grad_fn=<MmBackward0>),
 tensor([[-15.2409,  -6.2268]], grad_fn=<MmBackward0>),
 tensor([[-16.0551,  -6.6133]], grad_fn=<MmBackward0>),
 tensor([[-16.3224,  -7.7327]], grad_fn=<MmBackward0>),
 tensor([[-16.3601,  -8.5468]], grad_fn=<MmBackward0>),
 tensor([[-16.6374,  -9.0516]], grad_fn=<MmBackward0>),
 tensor([[-17.0240,  -9.2944]], grad_fn=<MmBackward0>),
 tensor([[-16.8785,  -9.1734]], grad_fn=<MmBackward0>),
 tensor([[-16.8143,  -8.8537]], grad_fn=<MmBackward0>),
 tensor([[-16.5444,  -8.0839]], grad_fn=<MmBackward0>),
 tensor([[-16.2349,  -7.5103]], grad_fn=<MmBackward0>),
 tensor([[-15.6999,  -6.3694]], grad_fn=<MmBackw